# DistilBERT Fine-Tuning on AG News — Google Colab Version

> **Note:** This notebook is optimized for Google Colab with GPU acceleration
> 
> Estimated training time: **5-10 minutes on GPU** (vs 30-60 min on CPU)

---

## Setup Instructions

1. **Enable GPU:**
   - Click **Runtime** → **Change runtime type** → Select **GPU** → **Save**

2. **Upload Files:**
   - Run Cell 1 to upload these 4 files:
     - `train_tensors.pt`
     - `val_tensors.pt`
     - `test_tensors.pt`
     - `prep_config.json`

3. **Run all cells** — training will start automatically

4. **Download Results:**
   - Run the final cell to download trained model and results

In [ ]:
# Cell 0 — Upload Files from Local Machine
from google.colab import files
import os

print("📤 UPLOAD FILES")
print("="*55)
print("Please upload these 4 files:")
print("  1. train_tensors.pt")
print("  2. val_tensors.pt")
print("  3. test_tensors.pt")
print("  4. prep_config.json")
print("="*55)

uploaded = files.upload()

print(f"\n✅ Uploaded files:")
for filename in uploaded.keys():
    size_mb = os.path.getsize(filename) / (1024**2)
    print(f"   {filename}: {size_mb:.1f} MB")

In [ ]:
# Cell 1 — Imports
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import (
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score
)

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"✅ Device: {device}")
if device.type == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print(f"   Running on CPU")

torch.manual_seed(42)
np.random.seed(42)

In [ ]:
# Cell 2 — Load from uploaded tensors
print("Loading preprocessed tensors...")

train_data = torch.load('train_tensors.pt', map_location='cpu')
val_data   = torch.load('val_tensors.pt', map_location='cpu')
test_data  = torch.load('test_tensors.pt', map_location='cpu')

with open('prep_config.json') as f:
    config = json.load(f)

BATCH_SIZE  = config['batch_size']
NUM_CLASSES = config['num_classes']
MAX_LENGTH  = config['max_length']
MODEL_NAME  = config['model_name']
LABEL_MAP   = {int(k): v for k, v in config['label_map'].items()}

print(f"✅ Tensors loaded")
print(f"   Train: {train_data['input_ids'].shape}")
print(f"   Val:   {val_data['input_ids'].shape}")
print(f"   Test:  {test_data['input_ids'].shape}")
print(f"\n   Model:      {MODEL_NAME}")
print(f"   Classes:    {NUM_CLASSES}")
print(f"   Max length: {MAX_LENGTH}")
print(f"   Batch size: {BATCH_SIZE}")

In [ ]:
# Cell 3 — Create DataLoaders
def make_dataloader(data_dict, batch_size, shuffle=False):
    dataset = TensorDataset(
        data_dict['input_ids'],
        data_dict['attention_mask'],
        data_dict['labels']
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=0)

train_loader = make_dataloader(train_data, BATCH_SIZE, shuffle=True)
val_loader   = make_dataloader(val_data,   BATCH_SIZE, shuffle=False)
test_loader  = make_dataloader(test_data,  BATCH_SIZE, shuffle=False)

print(f"✅ DataLoaders ready")
print(f"   Training batches:   {len(train_loader):,}")
print(f"   Validation batches: {len(val_loader):,}")
print(f"   Test batches:       {len(test_loader):,}")

In [ ]:
# Cell 4 — Load model
print("Loading DistilBERT model...")

model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_CLASSES,
    output_attentions=False,
    output_hidden_states=False
)

model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n✅ Model loaded")
print(f"   Total parameters:     {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")

In [ ]:
# Cell 5 — Training configuration
LEARNING_RATE = 2e-5
NUM_EPOCHS    = 6
WARMUP_RATIO  = 0.1

total_steps   = len(train_loader) * NUM_EPOCHS
warmup_steps  = int(total_steps * WARMUP_RATIO)

print(f"=== TRAINING CONFIGURATION ===")
print(f"Learning rate:  {LEARNING_RATE}")
print(f"Epochs:         {NUM_EPOCHS}")
print(f"Total steps:    {total_steps:,}")
print(f"Warmup steps:   {warmup_steps:,}")

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, eps=1e-8, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
criterion = nn.CrossEntropyLoss()

print(f"\n✅ Optimizer: AdamW")
print(f"✅ Scheduler: Linear warmup + decay")
print(f"✅ Loss: CrossEntropyLoss")

In [ ]:
# Cell 6 — Training function
def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss = 0
    total_correct = 0
    total_samples = 0

    for batch_idx, batch in enumerate(loader):
        input_ids, attention_mask, labels = [b.to(device) for b in batch]
        optimizer.zero_grad()

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        loss = criterion(logits, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = logits.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

        if (batch_idx + 1) % 100 == 0:
            avg_loss = total_loss / (batch_idx + 1)
            acc = total_correct / total_samples
            print(f"  Batch {batch_idx+1:>5,}/{len(loader):,} | Loss: {avg_loss:.4f} | Acc: {acc:.4f}")

    return total_loss / len(loader), total_correct / total_samples

In [ ]:
# Cell 7 — Evaluation function
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            input_ids, attention_mask, labels = [b.to(device) for b in batch]
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            loss = criterion(logits, labels)

            total_loss += loss.item()
            preds = logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    avg_loss = total_loss / len(loader)
    accuracy = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')

    return avg_loss, accuracy, f1, all_preds, all_labels

In [ ]:
# Cell 8 — Main training loop
print("="*55)
print("  STARTING DISTILBERT FINE-TUNING")
print("="*55)
print(f"  Epochs:    {NUM_EPOCHS}")
print(f"  Device:    {device}")
print(f"  Est. time: ~5-10 min on GPU")
print("="*55)

os.makedirs('models', exist_ok=True)
os.makedirs('logs', exist_ok=True)

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'val_f1': [], 'lr': []}
best_val_f1 = 0
best_epoch = 0
training_start = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_start = time.time()

    print(f"\n{'─'*55}")
    print(f"  EPOCH {epoch}/{NUM_EPOCHS}")
    print(f"{'─'*55}")

    print("  Training...")
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler, criterion, device)

    print("  Evaluating on validation set...")
    val_loss, val_acc, val_f1, _, _ = evaluate(model, val_loader, criterion, device)

    epoch_time = time.time() - epoch_start
    current_lr = scheduler.get_last_lr()[0]

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)
    history['lr'].append(current_lr)

    print(f"\n  Results Epoch {epoch}:")
    print(f"    Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"    Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f} | Val F1: {val_f1:.4f}")
    print(f"    LR: {current_lr:.2e} | Time: {epoch_time:.1f}s")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch = epoch
        model.save_pretrained('models/best_distilbert')
        print(f"    ✅ New best model saved! F1={val_f1:.4f}")

total_time = time.time() - training_start
print(f"\n{'='*55}")
print(f"  TRAINING COMPLETE")
print(f"  Total time: {total_time/60:.1f} min")
print(f"  Best epoch: {best_epoch}")
print(f"  Best val F1: {best_val_f1:.4f}")
print(f"{'='*55}")

with open('logs/training_history.json', 'w') as f:
    json.dump(history, f, indent=4)

In [ ]:
# Cell 9 — Training visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
epochs_range = range(1, NUM_EPOCHS + 1)

axes[0].plot(epochs_range, history['train_loss'], 'o-', color='#2196F3', label='Train Loss', linewidth=2)
axes[0].plot(epochs_range, history['val_loss'], 'o--', color='#F44336', label='Val Loss', linewidth=2)
axes[0].set_title('Loss Curves', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('CrossEntropy Loss')
axes[0].legend()
axes[0].set_xticks(epochs_range)

axes[1].plot(epochs_range, history['train_acc'], 'o-', color='#2196F3', label='Train Acc', linewidth=2)
axes[1].plot(epochs_range, history['val_acc'], 'o--', color='#F44336', label='Val Acc', linewidth=2)
axes[1].set_title('Accuracy Curves', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].set_xticks(epochs_range)

ax_f1 = axes[2]
ax_lr = ax_f1.twinx()
ax_f1.plot(epochs_range, history['val_f1'], 'o-', color='#4CAF50', label='Val F1', linewidth=2)
ax_lr.plot(epochs_range, history['lr'], 's--', color='#FF9800', label='Learning Rate', linewidth=1.5, alpha=0.7)
ax_f1.set_title('Validation F1 + Learning Rate', fontweight='bold')
ax_f1.set_xlabel('Epoch')
ax_f1.set_ylabel('F1 Score', color='#4CAF50')
ax_lr.set_ylabel('Learning Rate', color='#FF9800')
ax_f1.set_xticks(epochs_range)

lines1, labels1 = ax_f1.get_legend_handles_labels()
lines2, labels2 = ax_lr.get_legend_handles_labels()
ax_f1.legend(lines1+lines2, labels1+labels2, loc='lower right')

plt.suptitle('DistilBERT Fine-Tuning History', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('logs/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 10 — Load best model and evaluate on test
print("Loading best model for final evaluation...")

best_model = DistilBertForSequenceClassification.from_pretrained('models/best_distilbert')
best_model = best_model.to(device)

print("Evaluating on test set...")
test_loss, test_acc, test_f1, test_preds, test_labels = evaluate(best_model, test_loader, criterion, device)

print(f"\n{'='*55}")
print(f"  FINAL TEST RESULTS")
print(f"{'='*55}")
print(f"  Test Loss:     {test_loss:.4f}")
print(f"  Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"  Test F1:       {test_f1:.4f}")
print(f"{'='*55}")

label_names = ['World', 'Sports', 'Business', 'Sci/Tech']
print(f"\n{classification_report(test_labels, test_preds, target_names=label_names)}")

In [ ]:
# Cell 11 — Confusion matrix
plt.figure(figsize=(8, 6))

cm = confusion_matrix(test_labels, test_preds)
cm_pct = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues', xticklabels=label_names, yticklabels=label_names, cbar_kws={'label': 'Percentage (%)'})
plt.title(f'DistilBERT Confusion Matrix\nTest Accuracy: {test_acc*100:.2f}% | F1: {test_f1:.4f}', fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('logs/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 12 — Per-category analysis
print("=== PER-CATEGORY ANALYSIS ===\n")

for i, category in enumerate(label_names):
    mask = test_labels == i
    cat_acc = (test_preds[mask] == i).mean()
    cat_count = mask.sum()

    print(f"{category}:")
    print(f"  Test samples: {cat_count:,}")
    print(f"  Accuracy:     {cat_acc:.4f} ({cat_acc*100:.1f}%)")

    wrong_mask = (test_labels == i) & (test_preds != i)
    wrong_preds = test_preds[wrong_mask]
    if len(wrong_preds) > 0:
        from collections import Counter
        confused_with = Counter(wrong_preds)
        top_confusion = confused_with.most_common(1)[0]
        confused_label = label_names[top_confusion[0]]
        print(f"  Confused with: {confused_label} ({top_confusion[1]} times)")
    print()

In [ ]:
# Cell 13 — Save all results
final_results = {
    'model_name': MODEL_NAME,
    'num_epochs': NUM_EPOCHS,
    'best_epoch': best_epoch,
    'best_val_f1': round(best_val_f1, 4),
    'test_accuracy': round(test_acc, 4),
    'test_f1_macro': round(test_f1, 4),
    'test_loss': round(test_loss, 4),
    'training_config': {
        'learning_rate': LEARNING_RATE,
        'batch_size': BATCH_SIZE,
        'max_length': MAX_LENGTH,
        'optimizer': 'AdamW',
        'scheduler': 'linear_warmup',
        'warmup_ratio': WARMUP_RATIO
    },
    'per_class_accuracy': {label_names[i]: round(float((test_preds[test_labels==i] == i).mean()), 4) for i in range(NUM_CLASSES)}
}

with open('logs/bert_results.json', 'w') as f:
    json.dump(final_results, f, indent=4)

print("✅ Results saved to logs/bert_results.json")
print(json.dumps(final_results, indent=4))

---
## Download Results Back to Local System

Run the cell below to download trained model and results.

In [ ]:
# Cell 14 — Download Results
from google.colab import files
import shutil
import os

print("📥 PREPARING FILES FOR DOWNLOAD")
print("="*55)

# List files to download
files_to_download = [
    'logs/training_history.json',
    'logs/training_curves.png',
    'logs/confusion_matrix.png',
    'logs/bert_results.json',
]

print("\nDownloadable files:")
for f in files_to_download:
    if os.path.exists(f):
        size_mb = os.path.getsize(f) / (1024**2)
        print(f"  ✓ {f} ({size_mb:.1f} MB)")
        files.download(f)

print(f"\n✅ Downloaded {len(files_to_download)} files")

print("\n" + "="*55)
print("⚠️  FOR THE MODEL FOLDER:")
print("="*55)
print("\nThe 'models/best_distilbert/' folder is ~200MB.")
print("\nTo download it:")
print("1. Click the 'Files' icon in left panel (folder icon)")
print("2. Right-click 'models' folder → Download")
print("3. Extract to your local project at:")
print("   project1-document-ai/models/best_distilbert/")
print("\n" + "="*55)